### Imports

In [15]:
import json
import os
import pandas as pd

from utils_MS import *

# %load_ext autotime

In [16]:
# def run(params):

### Parameters

In [17]:
""" try:
    dir = os.path.dirname(os.path.abspath(__file__))
except:
    dir = os.getcwd()
print(dir) """

' try:\n    dir = os.path.dirname(os.path.abspath(__file__))\nexcept:\n    dir = os.getcwd()\nprint(dir) '

In [18]:
dict_dataset = {
    1: ["Mentos_2_process_NormalizationFiltered_format", ["Orange"]], # new mentos,  for Metabolomics
    2: ["deybis_filter_september_2br_3ar_format", ["SecoAmazonas"]],
    3: ["deybis_filter_december_2br_3ar_format", ["SecoAmazonas"]], # for Metabolomics
    4: ["deybis_filter_september_2br_10ar_format", ["SecoAmazonas"]],
    5: ["deybis_filter_december_2br_10ar_format", ["SecoAmazonas"]],
    6: ["deybis_filter_september_min_2br_3ar_format", ["SecoAmazonas"]],
    7: ["deybis_filter_december_min_2br_3ar_format", ["SecoAmazonas"]],
    8: ["vanessa_december_2br_3ar_format", ["SecoAmazonas"]],
}
dataset = dict_dataset[8] # change
dataset

['vanessa_december_2br_3ar_format', ['SecoAmazonas']]

In [19]:
params = {
    "exp": "exp8", # Change
    "methods": ["t-gae"], # ["vgae-base", "argva-base", "vgae-line", "dgi-tran", "t-gae"],
    "data_variations": ["none"],
    "has_transformation": False, # True or False
    "controls": dataset[1],
    "dimension": 32,
    "threshold_corr": 0.5,
    "threshold_log2": 0,
    "alpha": 0.05,
    "iterations": 1,
    "raw_data_file": dataset[0],
    "groups_id_no": ["Blank", "QC", "Std"],
    "sensitivity": False, # False: f1 (selectivity), True: f1 (selectivity), f2 (sensitivity)
    "obs": "",
    "seeds": [41, 42, 43, 44, 45, 46],
    
    "from": "python",
    "cuda": 1,
    "epochs": 100,
    "lr": 0.0001,
    "weight_decay": 1e-4,
    "patience": 10,
    "contamination": 0.1, # float in (0., 0.5)
    "n_jobs": 1, # -1 all
}

In [20]:
""" dir_path = "experiments/output"
res = sorted(os.listdir(dir_path))
n = len(res)
exp = "exp{}".format(n) """

exp = str(params["exp"])
exp

'exp8'

### Load dataset

In [21]:
# load dataset groups
if params["from"] == "python":
    df_raw = pd.read_csv("experiments/raw_data/{}.csv".format(params["raw_data_file"]), delimiter="|")
elif params["from"] == "drf":
    df_raw = pd.read_csv("{}".format(params["raw_data_file"]), delimiter="|") # from DRF
df_raw

,Alignment ID,Average Rt,Average Mz,Metabolite name,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,...,SecoCusco_1.3,SecoCusco_2.1,SecoCusco_2.2,SecoCusco_2.3,FrescoAmazonas_1.1,FrescoAmazonas_1.2,FrescoAmazonas_1.3,FrescoAmazonas_2.1,FrescoAmazonas_2.2,FrescoAmazonas_2.3
0,0,0.089,165.99460,NaN,1.213131e+06,1.177177e+06,1.566258e+05,1.731480e+06,1.699705e+06,6.280596e+06,...,3.338787e+05,1.133971e+06,1.808500e+05,3.342117e+05,4.322203e+06,1.821127e+06,1.770069e+06,1.743055e+06,1.463076e+06,6.428159e+06
1,1,0.090,158.01272,"[Similar to: 1-benzylhexahydropyrimidine-2,4,6...",1.431372e+04,2.609972e+04,5.571270e+03,4.865791e+04,3.923473e+04,8.001311e+03,...,4.652185e+03,3.434351e+04,2.178133e+04,3.596103e+03,5.631005e+04,4.148497e+04,9.098716e+03,2.515887e+04,5.974290e+04,5.789815e+03
2,2,0.097,172.95681,NaN,3.406202e+06,8.569343e+06,2.299997e+06,1.344103e+07,1.831757e+06,1.657309e+06,...,2.008602e+06,6.643525e+06,6.447431e+06,2.520397e+06,1.275967e+07,1.578762e+06,4.518729e+06,9.966807e+06,4.552709e+05,1.402781e+07
3,3,0.191,167.01326,NaN,5.020429e+07,4.827342e+07,1.113485e+06,6.889062e+07,6.861163e+07,2.800838e+07,...,2.189862e+06,4.159900e+07,4.101282e+07,1.728290e+06,7.507800e+07,7.443810e+07,3.294308e+07,6.365119e+07,6.403714e+07,2.607451e+07
4,4,0.196,165.98329,NaN,2.429038e+06,2.564504e+06,1.071655e+06,1.040565e+07,5.333877e+05,1.667311e+06,...,1.433978e+06,9.607175e+05,5.082268e+06,2.935405e+06,4.686568e+06,3.386707e+06,7.763971e+05,1.503685e+06,7.711962e+06,2.505585e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,554,29.965,341.26686,Docosahexaenoic acid,8.376403e+06,8.179919e+06,1.137834e+07,1.124024e+07,1.109478e+07,1.698218e+07,...,9.237098e+06,7.589148e+06,6.798410e+06,8.014118e+06,1.125906e+07,7.940634e+06,2.003224e+06,7.171052e+06,8.523936e+06,1.454813e+07
555,555,29.966,178.15937,N-Propylamphetamine,3.290299e+07,3.307227e+07,6.826276e+07,4.771023e+07,4.564733e+07,7.901293e+07,...,5.918501e+07,2.785011e+07,2.457465e+07,1.308641e+07,4.774097e+07,6.472719e+07,1.092961e+08,3.954838e+07,3.810093e+07,8.150158e+07
556,556,29.969,707.49253,NaN,1.537170e+07,1.921792e+07,4.989073e+06,2.964074e+07,2.545159e+07,5.389525e+06,...,2.580318e+06,1.344052e+07,1.132973e+07,1.192044e+06,3.228213e+07,2.365662e+07,3.800210e+06,2.567993e+07,6.028380e+06,1.913270e+06
557,557,29.969,165.11386,2-piperazinopyrimidine,2.308164e+06,2.275392e+06,6.454852e+06,1.710127e+06,3.178837e+06,8.738433e+06,...,5.209212e+06,1.847576e+06,1.788281e+06,1.986319e+06,3.774866e+06,3.512772e+06,1.176852e+07,2.692574e+06,2.835947e+06,7.170966e+06


### Format dataset

In [22]:
# has transformation
columns_data = list(df_raw.columns)[4:]
if params["has_transformation"]:
    print("transformation")
    for column in columns_data:
        df_raw[column] = df_raw[column].apply(lambda x: 10**x)
df_raw

,Alignment ID,Average Rt,Average Mz,Metabolite name,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,...,SecoCusco_1.3,SecoCusco_2.1,SecoCusco_2.2,SecoCusco_2.3,FrescoAmazonas_1.1,FrescoAmazonas_1.2,FrescoAmazonas_1.3,FrescoAmazonas_2.1,FrescoAmazonas_2.2,FrescoAmazonas_2.3
0,0,0.089,165.99460,NaN,1.213131e+06,1.177177e+06,1.566258e+05,1.731480e+06,1.699705e+06,6.280596e+06,...,3.338787e+05,1.133971e+06,1.808500e+05,3.342117e+05,4.322203e+06,1.821127e+06,1.770069e+06,1.743055e+06,1.463076e+06,6.428159e+06
1,1,0.090,158.01272,"[Similar to: 1-benzylhexahydropyrimidine-2,4,6...",1.431372e+04,2.609972e+04,5.571270e+03,4.865791e+04,3.923473e+04,8.001311e+03,...,4.652185e+03,3.434351e+04,2.178133e+04,3.596103e+03,5.631005e+04,4.148497e+04,9.098716e+03,2.515887e+04,5.974290e+04,5.789815e+03
2,2,0.097,172.95681,NaN,3.406202e+06,8.569343e+06,2.299997e+06,1.344103e+07,1.831757e+06,1.657309e+06,...,2.008602e+06,6.643525e+06,6.447431e+06,2.520397e+06,1.275967e+07,1.578762e+06,4.518729e+06,9.966807e+06,4.552709e+05,1.402781e+07
3,3,0.191,167.01326,NaN,5.020429e+07,4.827342e+07,1.113485e+06,6.889062e+07,6.861163e+07,2.800838e+07,...,2.189862e+06,4.159900e+07,4.101282e+07,1.728290e+06,7.507800e+07,7.443810e+07,3.294308e+07,6.365119e+07,6.403714e+07,2.607451e+07
4,4,0.196,165.98329,NaN,2.429038e+06,2.564504e+06,1.071655e+06,1.040565e+07,5.333877e+05,1.667311e+06,...,1.433978e+06,9.607175e+05,5.082268e+06,2.935405e+06,4.686568e+06,3.386707e+06,7.763971e+05,1.503685e+06,7.711962e+06,2.505585e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,554,29.965,341.26686,Docosahexaenoic acid,8.376403e+06,8.179919e+06,1.137834e+07,1.124024e+07,1.109478e+07,1.698218e+07,...,9.237098e+06,7.589148e+06,6.798410e+06,8.014118e+06,1.125906e+07,7.940634e+06,2.003224e+06,7.171052e+06,8.523936e+06,1.454813e+07
555,555,29.966,178.15937,N-Propylamphetamine,3.290299e+07,3.307227e+07,6.826276e+07,4.771023e+07,4.564733e+07,7.901293e+07,...,5.918501e+07,2.785011e+07,2.457465e+07,1.308641e+07,4.774097e+07,6.472719e+07,1.092961e+08,3.954838e+07,3.810093e+07,8.150158e+07
556,556,29.969,707.49253,NaN,1.537170e+07,1.921792e+07,4.989073e+06,2.964074e+07,2.545159e+07,5.389525e+06,...,2.580318e+06,1.344052e+07,1.132973e+07,1.192044e+06,3.228213e+07,2.365662e+07,3.800210e+06,2.567993e+07,6.028380e+06,1.913270e+06
557,557,29.969,165.11386,2-piperazinopyrimidine,2.308164e+06,2.275392e+06,6.454852e+06,1.710127e+06,3.178837e+06,8.738433e+06,...,5.209212e+06,1.847576e+06,1.788281e+06,1.986319e+06,3.774866e+06,3.512772e+06,1.176852e+07,2.692574e+06,2.835947e+06,7.170966e+06


In [23]:
# concat
df_join_raw = pd.concat([
    df_raw.iloc[:, :]], axis=1)
df_join_raw.set_index("Alignment ID", inplace=True)
df_join_raw

,Average Rt,Average Mz,Metabolite name,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,FrescoSanMartin_2.1,...,SecoCusco_1.3,SecoCusco_2.1,SecoCusco_2.2,SecoCusco_2.3,FrescoAmazonas_1.1,FrescoAmazonas_1.2,FrescoAmazonas_1.3,FrescoAmazonas_2.1,FrescoAmazonas_2.2,FrescoAmazonas_2.3
Alignment ID,,,,,,,,,,,,,,,,,,,,,
0,0.089,165.99460,NaN,1.213131e+06,1.177177e+06,1.566258e+05,1.731480e+06,1.699705e+06,6.280596e+06,1.696391e+06,...,3.338787e+05,1.133971e+06,1.808500e+05,3.342117e+05,4.322203e+06,1.821127e+06,1.770069e+06,1.743055e+06,1.463076e+06,6.428159e+06
1,0.090,158.01272,"[Similar to: 1-benzylhexahydropyrimidine-2,4,6...",1.431372e+04,2.609972e+04,5.571270e+03,4.865791e+04,3.923473e+04,8.001311e+03,3.985529e+04,...,4.652185e+03,3.434351e+04,2.178133e+04,3.596103e+03,5.631005e+04,4.148497e+04,9.098716e+03,2.515887e+04,5.974290e+04,5.789815e+03
2,0.097,172.95681,NaN,3.406202e+06,8.569343e+06,2.299997e+06,1.344103e+07,1.831757e+06,1.657309e+06,1.042663e+07,...,2.008602e+06,6.643525e+06,6.447431e+06,2.520397e+06,1.275967e+07,1.578762e+06,4.518729e+06,9.966807e+06,4.552709e+05,1.402781e+07
3,0.191,167.01326,NaN,5.020429e+07,4.827342e+07,1.113485e+06,6.889062e+07,6.861163e+07,2.800838e+07,6.561428e+07,...,2.189862e+06,4.159900e+07,4.101282e+07,1.728290e+06,7.507800e+07,7.443810e+07,3.294308e+07,6.365119e+07,6.403714e+07,2.607451e+07
4,0.196,165.98329,NaN,2.429038e+06,2.564504e+06,1.071655e+06,1.040565e+07,5.333877e+05,1.667311e+06,2.812425e+06,...,1.433978e+06,9.607175e+05,5.082268e+06,2.935405e+06,4.686568e+06,3.386707e+06,7.763971e+05,1.503685e+06,7.711962e+06,2.505585e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,29.965,341.26686,Docosahexaenoic acid,8.376403e+06,8.179919e+06,1.137834e+07,1.124024e+07,1.109478e+07,1.698218e+07,7.006518e+06,...,9.237098e+06,7.589148e+06,6.798410e+06,8.014118e+06,1.125906e+07,7.940634e+06,2.003224e+06,7.171052e+06,8.523936e+06,1.454813e+07
555,29.966,178.15937,N-Propylamphetamine,3.290299e+07,3.307227e+07,6.826276e+07,4.771023e+07,4.564733e+07,7.901293e+07,4.210922e+07,...,5.918501e+07,2.785011e+07,2.457465e+07,1.308641e+07,4.774097e+07,6.472719e+07,1.092961e+08,3.954838e+07,3.810093e+07,8.150158e+07
556,29.969,707.49253,NaN,1.537170e+07,1.921792e+07,4.989073e+06,2.964074e+07,2.545159e+07,5.389525e+06,2.626919e+07,...,2.580318e+06,1.344052e+07,1.132973e+07,1.192044e+06,3.228213e+07,2.365662e+07,3.800210e+06,2.567993e+07,6.028380e+06,1.913270e+06


In [24]:
# split
df_join_raw = df_join_raw.rename_axis(None)
# df_join_raw = df_join_raw.iloc[:, 2:]
df_join_raw

,Average Rt,Average Mz,Metabolite name,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,FrescoSanMartin_2.1,...,SecoCusco_1.3,SecoCusco_2.1,SecoCusco_2.2,SecoCusco_2.3,FrescoAmazonas_1.1,FrescoAmazonas_1.2,FrescoAmazonas_1.3,FrescoAmazonas_2.1,FrescoAmazonas_2.2,FrescoAmazonas_2.3
0,0.089,165.99460,NaN,1.213131e+06,1.177177e+06,1.566258e+05,1.731480e+06,1.699705e+06,6.280596e+06,1.696391e+06,...,3.338787e+05,1.133971e+06,1.808500e+05,3.342117e+05,4.322203e+06,1.821127e+06,1.770069e+06,1.743055e+06,1.463076e+06,6.428159e+06
1,0.090,158.01272,"[Similar to: 1-benzylhexahydropyrimidine-2,4,6...",1.431372e+04,2.609972e+04,5.571270e+03,4.865791e+04,3.923473e+04,8.001311e+03,3.985529e+04,...,4.652185e+03,3.434351e+04,2.178133e+04,3.596103e+03,5.631005e+04,4.148497e+04,9.098716e+03,2.515887e+04,5.974290e+04,5.789815e+03
2,0.097,172.95681,NaN,3.406202e+06,8.569343e+06,2.299997e+06,1.344103e+07,1.831757e+06,1.657309e+06,1.042663e+07,...,2.008602e+06,6.643525e+06,6.447431e+06,2.520397e+06,1.275967e+07,1.578762e+06,4.518729e+06,9.966807e+06,4.552709e+05,1.402781e+07
3,0.191,167.01326,NaN,5.020429e+07,4.827342e+07,1.113485e+06,6.889062e+07,6.861163e+07,2.800838e+07,6.561428e+07,...,2.189862e+06,4.159900e+07,4.101282e+07,1.728290e+06,7.507800e+07,7.443810e+07,3.294308e+07,6.365119e+07,6.403714e+07,2.607451e+07
4,0.196,165.98329,NaN,2.429038e+06,2.564504e+06,1.071655e+06,1.040565e+07,5.333877e+05,1.667311e+06,2.812425e+06,...,1.433978e+06,9.607175e+05,5.082268e+06,2.935405e+06,4.686568e+06,3.386707e+06,7.763971e+05,1.503685e+06,7.711962e+06,2.505585e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,29.965,341.26686,Docosahexaenoic acid,8.376403e+06,8.179919e+06,1.137834e+07,1.124024e+07,1.109478e+07,1.698218e+07,7.006518e+06,...,9.237098e+06,7.589148e+06,6.798410e+06,8.014118e+06,1.125906e+07,7.940634e+06,2.003224e+06,7.171052e+06,8.523936e+06,1.454813e+07
555,29.966,178.15937,N-Propylamphetamine,3.290299e+07,3.307227e+07,6.826276e+07,4.771023e+07,4.564733e+07,7.901293e+07,4.210922e+07,...,5.918501e+07,2.785011e+07,2.457465e+07,1.308641e+07,4.774097e+07,6.472719e+07,1.092961e+08,3.954838e+07,3.810093e+07,8.150158e+07
556,29.969,707.49253,NaN,1.537170e+07,1.921792e+07,4.989073e+06,2.964074e+07,2.545159e+07,5.389525e+06,2.626919e+07,...,2.580318e+06,1.344052e+07,1.132973e+07,1.192044e+06,3.228213e+07,2.365662e+07,3.800210e+06,2.567993e+07,6.028380e+06,1.913270e+06
557,29.969,165.11386,2-piperazinopyrimidine,2.308164e+06,2.275392e+06,6.454852e+06,1.710127e+06,3.178837e+06,8.738433e+06,1.248976e+06,...,5.209212e+06,1.847576e+06,1.788281e+06,1.986319e+06,3.774866e+06,3.512772e+06,1.176852e+07,2.692574e+06,2.835947e+06,7.170966e+06


In [25]:
# get groups name
groups_id_no = params["groups_id_no"]
groups_id = []
for item in df_join_raw.iloc[:, 3:].columns.values:
    group_id = item.split("_")[0]
    if group_id not in groups_id and group_id not in groups_id_no:
        groups_id.append(group_id)
groups_id

['SecoAmazonas',
 'FrescoSanMartin',
 'FrescoCusco',
 'SecoSanMartin',
 'SecoCusco',
 'FrescoAmazonas']

In [26]:
# delete no sample columns
""" columns_delete = [columna for columna in df_join_raw.columns if columna.split("_")[0] in columns_no_sample]
df_join_raw.drop(columns_delete, axis=1, inplace=True)
df_join_raw """

' columns_delete = [columna for columna in df_join_raw.columns if columna.split("_")[0] in columns_no_sample]\ndf_join_raw.drop(columns_delete, axis=1, inplace=True)\ndf_join_raw '

In [27]:
# get subgroups names

""" def get_subgroups_id(df_join_raw, groups, by_group=False):
    dict_groups_id = {}
    for group in groups:
        # get group
        if by_group:
            dict_groups_id[group] = ["1"]
        else:
            columns = list(df_join_raw.filter(like=group).columns)
            subgroups = [item.split("{}_".format(group))[1].split(".")[0] for item in columns]
            subgroups = np.unique(subgroups)
            dict_groups_id[group] = subgroups.tolist()
    return dict_groups_id """

subgroups_id = get_subgroups_id(df_join_raw, groups_id)
subgroups_id

{'SecoAmazonas': ['1', '2'],
 'FrescoSanMartin': ['1', '2'],
 'FrescoCusco': ['1', '2'],
 'SecoSanMartin': ['1', '2'],
 'SecoCusco': ['1', '2'],
 'FrescoAmazonas': ['1', '2']}

In [28]:
# count analtical repetitions
# df_join_raw.filter(like="AA_1.")

In [29]:
# check distribution

In [30]:
""" x = df_join_raw.iloc[2, 3:]
print(x.min(), x.max(), x.mean())
x.hist(bins=200) """

' x = df_join_raw.iloc[2, 3:]\nprint(x.min(), x.max(), x.mean())\nx.hist(bins=200) '

In [31]:
# f_join_raw.iloc[:, 5].hist(bins=100)

In [32]:
params["controls"], groups_id

(['SecoAmazonas'],
 ['SecoAmazonas',
  'FrescoSanMartin',
  'FrescoCusco',
  'SecoSanMartin',
  'SecoCusco',
  'FrescoAmazonas'])

In [33]:
# get groups combination
groups = []
controls = params["controls"]

groups = []
for control in controls:
    for group_id in groups_id:
        if control != group_id:
            groups.append([control, group_id])
print(groups)

[['SecoAmazonas', 'FrescoSanMartin'], ['SecoAmazonas', 'FrescoCusco'], ['SecoAmazonas', 'SecoSanMartin'], ['SecoAmazonas', 'SecoCusco'], ['SecoAmazonas', 'FrescoAmazonas']]


### Create folders

In [34]:
# create experiments folder
try: 
    os.mkdir("experiments/output/{}".format(exp))
    os.mkdir("experiments/output/{}/correlations".format(exp))
    os.mkdir("experiments/output/{}/preprocessing".format(exp))
    os.mkdir("experiments/output/{}/preprocessing/edges".format(exp))
    os.mkdir("experiments/output/{}/preprocessing/graphs_data".format(exp))
    os.mkdir("experiments/output/{}/loss".format(exp))
    os.mkdir("experiments/output/{}/node_embeddings".format(exp))
    os.mkdir("experiments/output/{}/common_nodes".format(exp))
    os.mkdir("experiments/output/{}/filter_raw".format(exp))
    os.mkdir("experiments/output/{}/plots".format(exp))
except OSError as error: 
    print(error)

### Save dataset and parameters

In [35]:
# save dataset
df_join_raw.to_csv("experiments/input/{}_raw.csv".format(exp), index=True)

# save parameters
parameters = {
    "exp": exp,
    "methods": params["methods"],
    "data_variations": params["data_variations"],
    "has_transformation": params["has_transformation"],
    "controls": params["controls"],
    "dimension": params["dimension"],
    "threshold_corr": params["threshold_corr"],
    "threshold_log2": params["threshold_log2"],
    "alpha": params["alpha"],
    "iterations": params["iterations"],
    "raw_data_file": params["raw_data_file"],
    "groups_id": groups_id,
    "subgroups_id": subgroups_id,
    "groups": groups,
    "groups_id_no": params["groups_id_no"],
    "sensitivity": params["sensitivity"],
    
    "from": params["from"],
    "cuda": params["cuda"],
    "epochs": params["epochs"],
    "lr": params["lr"],
    "weight_decay": params["weight_decay"],
    "patience": params["patience"],
    "contamination": params["contamination"],
    "n_jobs": params["n_jobs"],

    "seeds": params["seeds"],
    "obs": params["obs"]
}

with open("experiments/output/{}/parameters.json".format(exp), "w") as outfile:
    json.dump(parameters, outfile, indent=4)

In [36]:
experiments = {
    "exp": exp
}

with open("exp.json".format(experiments), "w") as outfile:
    json.dump(experiments, outfile, indent=4)

In [37]:
# return exp